In [ ]:
# simplify wetlands

In [1]:
#!/usr/bin/env python3
"""
simplify_wetlands_for_portal.py

Create a simplified version of wetlands_portal.geojson for faster web mapping.

- Input:  portal_data/wetlands_portal.geojson
- Output: portal_data/wetlands_portal_simplified.geojson

Simplification is done in EPSG:5070 (meters), then converted back to WGS84.
"""

from pathlib import Path
import geopandas as gpd
import os

# --------------------------------------------------
# USER PATHS
# --------------------------------------------------

PROJECT_ROOT = Path(os.environ.get("DML_PROJECT_ROOT", "."))

PORTAL_DIR = PROJECT_ROOT / "portal_data"
IN_GEOJSON = PORTAL_DIR / "wetlands_portal.geojson"
OUT_GEOJSON = PORTAL_DIR / "wetlands_portal_simplified.geojson"

# --------------------------------------------------
# SETTINGS
# --------------------------------------------------

# Work CRS for simplification (meters)
WORK_CRS = "EPSG:5070"
WEB_CRS = "EPSG:4326"

# Geometric simplification tolerance in meters
# Try 50 m to start; if still slow, you can bump to 75–100 m.
TOLERANCE_METERS = 50.0


def main():
    print(f"[INFO] Reading input GeoJSON:\n  {IN_GEOJSON}")
    gdf = gpd.read_file(IN_GEOJSON)
    print(f"[INFO] Features read: {len(gdf)}")

    # Ensure we have a CRS
    if gdf.crs is None:
        # Assume current coords are already in WGS84
        print("[WARN] Input has no CRS. Assuming EPSG:4326 (WGS84).")
        gdf = gdf.set_crs(WEB_CRS)

    # Reproject to WORK_CRS for simplification in meters
    if str(gdf.crs) != WORK_CRS:
        print(f"[INFO] Reprojecting from {gdf.crs} to {WORK_CRS} for simplification.")
        gdf = gdf.to_crs(WORK_CRS)

    # Simplify geometries
    print(f"[INFO] Simplifying geometries with tolerance={TOLERANCE_METERS} m.")
    gdf["geometry"] = gdf.geometry.simplify(
        TOLERANCE_METERS,
        preserve_topology=True
    )

    # Reproject back to WGS84 for web use
    print(f"[INFO] Reprojecting simplified geometries to {WEB_CRS}.")
    gdf_web = gdf.to_crs(WEB_CRS)

    # Write output
    OUT_GEOJSON.parent.mkdir(parents=True, exist_ok=True)
    print(f"[INFO] Writing simplified GeoJSON:\n  {OUT_GEOJSON}")
    gdf_web.to_file(OUT_GEOJSON, driver="GeoJSON")

    print("[INFO] Done.")


if __name__ == "__main__":
    main()


[INFO] Reading input GeoJSON:
  /Users/kimberlyvanmeter/Library/CloudStorage/GoogleDrive-vanmeterlab@gmail.com/My Drive/Research/Projects/NASA - UMRB Legacy Wetlands/portal_data/wetlands_portal.geojson
[INFO] Features read: 200386
[INFO] Reprojecting from EPSG:4326 to EPSG:5070 for simplification.
[INFO] Simplifying geometries with tolerance=50.0 m.


/opt/anaconda3/envs/CBP_random_forest/lib/python3.12/site-packages/shapely/constructive.py:863: RuntimeWarning: invalid value encountered in simplify_preserve_topology
  return lib.simplify_preserve_topology(geometry, tolerance, **kwargs)


[INFO] Reprojecting simplified geometries to EPSG:4326.
[INFO] Writing simplified GeoJSON:
  /Users/kimberlyvanmeter/Library/CloudStorage/GoogleDrive-vanmeterlab@gmail.com/My Drive/Research/Projects/NASA - UMRB Legacy Wetlands/portal_data/wetlands_portal_simplified.geojson
[INFO] Done.
